In [ ]:
import sys
sys.path.insert(0, '.')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from tqdm import tqdm

from src.data_loader import load_config, load_btc_5min, split_is_oos
from src.backtest_engine import run_backtest, compute_stats, parameter_sweep

# === Load config and data ===
config = load_config()
df_signals = pd.read_csv("Data/df_signals_v2.csv", parse_dates=["Date", "Expiry"])
df_5m = load_btc_5min(config)

# === IS/OOS split ===
df_is, df_oos = split_is_oos(df_signals, config, date_col='Date')
print(f"Signals: {len(df_signals)} total, IS={len(df_is)}, OOS={len(df_oos)}")

# === Parameters to sweep (from config) ===
gap_threshold_values = config['backtest']['gap_thresholds']
sl_multiplier_values = config['backtest']['sl_multipliers']
spread_tolerance_values = config['backtest']['spread_tolerances']
holding_period_values = config['backtest']['holding_periods']

print(f"Sweep grid: {len(gap_threshold_values)} gaps × {len(sl_multiplier_values)} SL "
      f"× {len(spread_tolerance_values)} spread × {len(holding_period_values)} HP "
      f"= {len(gap_threshold_values)*len(sl_multiplier_values)*len(spread_tolerance_values)*len(holding_period_values)} combos")

# === Run sweep on IS ===
print("\nRunning parameter sweep on IS...")
results = {}
combinations = [(gap, sl, spread, hp) 
                for gap in gap_threshold_values
                for sl in sl_multiplier_values
                for spread in spread_tolerance_values
                for hp in holding_period_values]

for gap, sl, spread, hp in tqdm(combinations, desc="IS Sweep"):
    trades = run_backtest(df_is, df_5m, config,
                          gap_threshold=gap, sl_multiplier=sl,
                          spread_tolerance=spread, holding_period=hp)
    results[(gap, sl, spread, hp)] = trades

print(f"\nCompleted: {len(results)} combinations")

In [ ]:
# compute_stats is now imported from src.backtest_engine
# Just verify it works
test_params = list(results.keys())[0]
test_stats = compute_stats(results[test_params])
print(f"Test compute_stats on {test_params}: {test_stats}")

In [ ]:
# Compute statistics for each combination
summary = []

for (gap, sl, spread, hp), df in results.items():
    stats = compute_stats(df.copy())
    stats["gap_threshold"] = gap
    stats["sl_multiplier"] = sl
    stats["spread_tolerance"] = spread
    stats["holding_period"] = hp
    stats["sample"] = "IS"
    summary.append(stats)

# Final DataFrame sorted by Sharpe
df_summary = pd.DataFrame(summary)
df_summary = df_summary.sort_values(by="Sharpe", ascending=False).reset_index(drop=True)

print(f"Total combinations: {len(df_summary)}")
print(f"Combinations with Sharpe > 0: {(df_summary['Sharpe'] > 0).sum()}")
print(f"\nTop 10 by Sharpe:")
df_summary.head(10)

In [ ]:
best_combo = df_summary.iloc[0]  # sorted by Sharpe
print("Best combination (by Sharpe):")
print(best_combo)

In [ ]:
best_total_return = df_summary.sort_values("Total_Return_Pct", ascending=False).iloc[0]
print("Best Total Return:")
print(best_total_return)

In [ ]:
best_cagr = df_summary.sort_values("CAGR", ascending=False).iloc[0]
print("Best CAGR:")
print(best_cagr)

In [ ]:
best_max_dd = df_summary.sort_values("Max_Drawdown", ascending=False).iloc[0]
print("Lowest Max Drawdown:")
print(best_max_dd)

In [ ]:
# Compute mean and standard deviation of key metrics
metrics = ["Sharpe", "Total_Return_Pct", "CAGR", "Max_Drawdown", "Win_Rate_Pct", "N_Trades"]
df_summary[metrics].describe().round(2)

In [ ]:
# Robust filter (updated with holding_period)
robuste = df_summary[
    (df_summary["Sharpe"] > config['robustness']['min_sharpe']) &
    (df_summary["Total_Return_Pct"] > config['robustness']['min_total_return_pct']) &
    (df_summary["CAGR"] > config['robustness']['min_cagr_pct']) &
    (df_summary["Max_Drawdown"] > config['robustness']['max_drawdown_pct']) &
    (df_summary["N_Trades"] >= 30)  # relaxed from 200 since multi-day HP means fewer trades
]

print(f"Robust combinations found: {len(robuste)}")
print(f"\nBreakdown by holding_period:")
print(robuste.groupby("holding_period").agg(
    n_combos=("Sharpe", "count"),
    median_sharpe=("Sharpe", "median"),
    median_wr=("Win_Rate_Pct", "median"),
    median_n_trades=("N_Trades", "median"),
).to_string())
print(f"\nTop 5 robust:")
robuste.sort_values("Sharpe", ascending=False).head(5)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

plt.figure(figsize=(12, 6))

n = len(robuste)
mid_index = n // 2

for i, (idx, row) in enumerate(robuste.iterrows()):
    key = (row["gap_threshold"], row["sl_multiplier"], row["spread_tolerance_pct"])
    df = results[key].copy()
    df["Entry_Date"] = pd.to_datetime(df["Entry_Date"])
    df = df.sort_values("Entry_Date")
    df["Equity"] = (df["Return_Pct"] / 100 + 1).cumprod()
    label = f"{key} - Sharpe {row['Sharpe']:.2f}"

    # Custom colors
    if i == 0:
        color = "green"
        linewidth = 2.5
    elif i == n - 1:
        color = "red"
        linewidth = 2.5
    elif i == mid_index:
        color = "blue"
        linewidth = 2.5
    else:
        # Shades of gray for the others
        gray_level = 0.3 + 0.5 * (i / n)  # from dark gray to medium
        color = (gray_level, gray_level, gray_level)
        linewidth = 1.2

    plt.plot(df["Entry_Date"], df["Equity"], label=label, color=color, linewidth=linewidth)

plt.title("Equity Lines of Robust Strategies")
plt.xlabel("Date")
plt.ylabel("Equity Line (base = 1)")
plt.grid(True)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# === Loop over all robust strategies ===
for idx, row in robuste.iterrows():
    params = (row["gap_threshold"], row["sl_multiplier"], row["spread_tolerance_pct"])
    
    if params not in results:
        print(f"Strategy {params} not present in results, skipping.")
        continue

    df = results[params].copy()
    df["Entry_Date"] = pd.to_datetime(df["Entry_Date"])
    df["Entry_Timestamp"] = pd.to_datetime(df["Entry_Timestamp"])
    df["Exit_Timestamp"] = pd.to_datetime(df["Exit_Timestamp"])
    df = df.sort_values("Entry_Date")
    df["Equity"] = (df["Return_Pct"] / 100 + 1).cumprod()

    # === General statistics ===
    n_trades = len(df)
    win_rate = (df["Return_Pct"] > 0).mean() * 100
    returns = df["Return_Pct"] / 100
    n_days = (df["Exit_Timestamp"].max() - df["Entry_Timestamp"].min()).days + 1
    avg_return = returns.mean()
    std_return = returns.std()
    trades_per_year = len(df) / n_days * 365
    sharpe = avg_return / std_return * np.sqrt(trades_per_year) if std_return > 0 else 0
    df["Duration"] = df["Exit_Timestamp"] - df["Entry_Timestamp"]
    avg_duration = df["Duration"].mean()
    ret_stats = df["Return_Pct"].describe()

    # Compute average duration formatted
    days = avg_duration.days
    hours, remainder = divmod(avg_duration.seconds, 3600)
    minutes, _ = divmod(remainder, 60)

    # === Breakdown by direction and exit type ===
    by_dir = df.groupby("Direction")["Return_Pct"].agg(["count", "mean", "std", "min", "max"])
    by_exit = df.groupby("Exit_Type")["Return_Pct"].agg(["count", "mean", "std", "min", "max"])

    # === Text output ===
    print(f"\nStrategy analysis {params}")
    print(f"- Number of trades: {n_trades}")
    print(f"- Win rate: {win_rate:.2f}%")
    print(f"- Sharpe (annualized): {sharpe:.2f}")
    print(f"- Average trade duration: {days} days, {hours} hours and {minutes} minutes")
    print(f"\n→ Return statistics:\n{ret_stats}")
    print(f"\n→ Breakdown by direction:\n{by_dir}")
    print(f"\n→ Breakdown by exit type:\n{by_exit}")

    # === Plots ===
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"Strategy {params} - Sharpe {row['Sharpe']:.2f}", fontsize=14)

    # 1. Distribution of returns
    sns.histplot(df["Return_Pct"], bins=30, kde=True, ax=axs[0, 0], color="skyblue")
    axs[0, 0].set_title("Distribution of Returns (%)")

    # 2. Equity line
    axs[0, 1].plot(df["Entry_Date"], df["Equity"], color="green")
    axs[0, 1].set_title("Equity Line")
    axs[0, 1].set_xlabel("Data")
    axs[0, 1].set_ylabel("Equity (base = 1)")

    # 3. Average returns by exit type
    by_exit["mean"].plot(kind="bar", ax=axs[1, 0], color="orange", title="Average Return by Exit Type")
    axs[1, 0].set_ylabel("Return (%)")

    # 4. Average returns by direction
    by_dir["mean"].plot(kind="bar", ax=axs[1, 1], color="purple", title="Average Return by Direction")
    axs[1, 1].set_ylabel("Return (%)")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
import numpy as np
import pandas as pd

# List to gather results
score_data = []

for idx, row in robuste.iterrows():
    params = (row["gap_threshold"], row["sl_multiplier"], row["spread_tolerance"], row["holding_period"])
    
    if params not in results:
        print(f"Strategy {params} not present in results, skipping.")
        continue

    df = results[params].copy()
    if df.empty:
        continue
    
    df["Entry_Timestamp"] = pd.to_datetime(df["Entry_Timestamp"])
    df["Exit_Timestamp"] = pd.to_datetime(df["Exit_Timestamp"])
    df = df.sort_values("Entry_Timestamp")

    # === Equity and cumulative returns ===
    df["Equity"] = (df["Return_Pct"] / 100 + 1).cumprod()

    # === Max drawdown calculation ===
    equity = df["Equity"]
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max
    max_drawdown = drawdown.min() * 100

    # === Sharpe ===
    returns = df["Return_Pct"] / 100
    n_days = (df["Exit_Timestamp"].max() - df["Entry_Timestamp"].min()).days + 1
    avg_return = returns.mean()
    std_return = returns.std()
    trades_per_year = len(df) / n_days * 365 if n_days > 0 else 0
    sharpe = avg_return / std_return * np.sqrt(trades_per_year) if std_return > 0 else 0

    # === Other metrics ===
    win_rate = (df["Return_Pct"] > 0).mean()
    n_trades = len(df)

    # === TP Accuracy ===
    tp_rate = (df["Exit_Type"] == "TP").mean()
    df["Error_TP"] = df["Spot_Exit"] - df["Fair_Value"]
    rmse_error_tp = np.sqrt((df["Error_TP"] ** 2).mean())

    # === Breakdown by direction ===
    dir_stats = df.groupby("Direction")["Return_Pct"].mean()
    mean_return_LONG = dir_stats.get("LONG", 0)
    mean_return_SHORT = dir_stats.get("SHORT", 0)
    min_direction_return = min(mean_return_LONG, mean_return_SHORT)

    # === Breakdown by exit type ===
    exit_stats = df.groupby("Exit_Type")["Return_Pct"].mean()
    mean_return_SL = exit_stats.get("SL", 0)

    # === Append raw metrics ===
    score_data.append({
        "params": params,
        "Sharpe": sharpe,
        "WinRate": win_rate,
        "Num_Trades": n_trades,
        "TP_Rate": tp_rate,
        "RMSE_TP": rmse_error_tp,
        "Min_Dir_Return": min_direction_return,
        "Mean_SL_Return": mean_return_SL,
        "Max_Drawdown": max_drawdown,
    })

# === BUILD df_scores ===
df_scores = pd.DataFrame(score_data)

if df_scores.empty:
    print("No robust strategies found for scoring.")
else:
    # === Normalize and compute composite score ===
    from sklearn.preprocessing import MinMaxScaler
    
    cols_to_max = ["Sharpe", "WinRate", "Num_Trades", "TP_Rate", "Min_Dir_Return"]
    cols_to_min = ["RMSE_TP", "Mean_SL_Return", "Max_Drawdown"]
    
    scaler = MinMaxScaler()
    for col in cols_to_max:
        if col in df_scores.columns:
            vals = df_scores[col].values.reshape(-1, 1)
            df_scores[f"{col}_norm"] = scaler.fit_transform(vals)
    
    for col in cols_to_min:
        if col in df_scores.columns:
            vals = (-df_scores[col].values).reshape(-1, 1)
            df_scores[f"{col}_norm"] = scaler.fit_transform(vals)
    
    # Weighted composite score
    weights = config['scoring']
    df_scores["Score"] = (
        df_scores.get("Sharpe_norm", 0) * weights['sharpe_weight'] +
        df_scores.get("Max_Drawdown_norm", 0) * weights['max_dd_weight'] +
        df_scores.get("TP_Rate_norm", 0) * weights['tp_rate_weight'] +
        df_scores.get("WinRate_norm", 0) * weights['win_rate_weight'] +
        df_scores.get("Mean_SL_Return_norm", 0) * weights['mean_sl_return_weight'] +
        df_scores.get("Min_Dir_Return_norm", 0) * weights['min_dir_return_weight'] +
        df_scores.get("Num_Trades_norm", 0) * weights['num_trades_weight'] +
        df_scores.get("RMSE_TP_norm", 0) * weights['rmse_tp_weight']
    ) * 100
    
    df_scores = df_scores.sort_values("Score", ascending=False).reset_index(drop=True)
    print(f"Scored {len(df_scores)} strategies")
    print(f"\nTop 5 by composite score:")
    print(df_scores[["params", "Score", "Sharpe", "WinRate", "Num_Trades", "TP_Rate"]].head(5).to_string(index=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if not df_scores.empty:
    # Extract parameters (now 4-tuple)
    df_scores["gap"] = df_scores["params"].apply(lambda x: x[0])
    df_scores["sl_mult"] = df_scores["params"].apply(lambda x: x[1])
    df_scores["spread_tol"] = df_scores["params"].apply(lambda x: x[2])
    df_scores["hp"] = df_scores["params"].apply(lambda x: x[3])

    # Heatmaps for each holding period
    fig, axes = plt.subplots(1, len(holding_period_values), figsize=(6*len(holding_period_values), 5))
    if len(holding_period_values) == 1:
        axes = [axes]
    
    for ax, hp in zip(axes, holding_period_values):
        subset = df_scores[(df_scores["gap"] == 0.02) & (df_scores["hp"] == hp)]
        if not subset.empty:
            pivot = subset.pivot_table(index="sl_mult", columns="spread_tol", values="Score", aggfunc="mean")
            sns.heatmap(pivot, annot=True, cmap="viridis", fmt=".1f", ax=ax)
            ax.set_title(f"Score (gap=0.02, HP={hp}d)")
            ax.set_xlabel("Spread Tolerance")
            ax.set_ylabel("SL Multiplier")
    
    plt.tight_layout()
    plt.show()
    
    # Score distribution by HP
    fig, ax = plt.subplots(figsize=(10, 5))
    df_scores.boxplot(column="Score", by="hp", ax=ax)
    ax.set_title("Composite Score by Holding Period")
    ax.set_xlabel("Holding Period (days)")
    ax.set_ylabel("Score")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(df_scores["RMSE_TP"], df_scores["Score"], c="red", label="Score vs RMSE")
plt.scatter(df_scores["TP_Rate"], df_scores["Score"], c="blue", label="Score vs TP Rate")
plt.xlabel("Value")
plt.ylabel("Score")
plt.legend()
plt.title("Relationship between Score and Metrics")
plt.grid(True)
plt.show()

In [ ]:
# Top 3 selected strategies
if not df_scores.empty:
    top_3 = df_scores.sort_values(by="Score", ascending=False).head(3)
    print("Top 3 selected strategies:\n")
    print(top_3[[
        "params", "Score", "Sharpe", "WinRate", "Num_Trades", "TP_Rate",
        "RMSE_TP", "Min_Dir_Return", "Mean_SL_Return", "Max_Drawdown"
    ]].to_string(index=False))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

if not df_scores.empty:
    top_3 = df_scores.sort_values(by="Score", ascending=False).head(3)
    
    # === OOS Validation of Top 3 ===
    print("=== OOS VALIDATION OF TOP 3 ===\n")
    df_comparativa = []
    
    for _, row in top_3.iterrows():
        params = row["params"]
        gap, sl, spread, hp = params
        
        # IS stats
        is_stats = compute_stats(results[params])
        
        # OOS backtest
        trades_oos = run_backtest(df_oos, df_5m, config,
                                  gap_threshold=gap, sl_multiplier=sl,
                                  spread_tolerance=spread, holding_period=int(hp))
        oos_stats = compute_stats(trades_oos)
        
        sharpe_ratio = oos_stats['Sharpe'] / is_stats['Sharpe'] if is_stats['Sharpe'] != 0 else 0
        
        df_comparativa.append({
            "Strategy": str(params),
            "IS_Sharpe": is_stats['Sharpe'],
            "IS_WR": is_stats['Win_Rate_Pct'],
            "IS_Return": is_stats['Total_Return_Pct'],
            "IS_N": is_stats['N_Trades'],
            "OOS_Sharpe": oos_stats['Sharpe'],
            "OOS_WR": oos_stats['Win_Rate_Pct'],
            "OOS_Return": oos_stats['Total_Return_Pct'],
            "OOS_N": oos_stats['N_Trades'],
            "Sharpe_OOS_IS_Ratio": sharpe_ratio,
        })
        
        print(f"Strategy {params}:")
        print(f"  IS:  Sharpe={is_stats['Sharpe']:.2f}, WR={is_stats['Win_Rate_Pct']:.1f}%, "
              f"Return={is_stats['Total_Return_Pct']:.1f}%, N={is_stats['N_Trades']}")
        print(f"  OOS: Sharpe={oos_stats['Sharpe']:.2f}, WR={oos_stats['Win_Rate_Pct']:.1f}%, "
              f"Return={oos_stats['Total_Return_Pct']:.1f}%, N={oos_stats['N_Trades']}")
        print(f"  Sharpe OOS/IS: {sharpe_ratio:.2f}")
        print()
    
    df_comparativa = pd.DataFrame(df_comparativa)
    
    # Plotly comparison chart
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Sharpe IS vs OOS", "Return IS vs OOS"))
    for i, row in df_comparativa.iterrows():
        name = f"S{i+1}"
        fig.add_trace(go.Bar(name=f"{name} IS", x=[name], y=[row["IS_Sharpe"]], marker_color="steelblue"), row=1, col=1)
        fig.add_trace(go.Bar(name=f"{name} OOS", x=[name], y=[row["OOS_Sharpe"]], marker_color="coral"), row=1, col=1)
        fig.add_trace(go.Bar(name=f"{name} IS", x=[name], y=[row["IS_Return"]], marker_color="steelblue", showlegend=False), row=1, col=2)
        fig.add_trace(go.Bar(name=f"{name} OOS", x=[name], y=[row["OOS_Return"]], marker_color="coral", showlegend=False), row=1, col=2)
    fig.update_layout(title="Top 3 Strategies: IS vs OOS Comparison", barmode="group", height=400)
    fig.show()

In [ ]:
# Comparative analysis already done in the cell above
# Show the comparison table
if 'df_comparativa' in dir() and not df_comparativa.empty:
    print("=== IS vs OOS Comparison Table ===")
    print(df_comparativa.to_string(index=False))

In [ ]:
# Display top 3 details
if 'df_comparativa' in dir() and not df_comparativa.empty:
    print("Top 3 strategies with OOS validation:")
    for i, row in df_comparativa.iterrows():
        ratio = row['Sharpe_OOS_IS_Ratio']
        status = "PASS" if ratio > 0 else "FAIL"
        print(f"  [{status}] {row['Strategy']}: OOS/IS ratio = {ratio:.2f}")

In [ ]:
# === Save results (v2) ===
import pickle

# df_trades for each combination (4-tuple keys now)
with open("Data/results_v2_sweep.pkl", "wb") as f:
    pickle.dump(results, f)

# Sweep summary CSV
df_summary.to_csv("Data/sweep_IS_v2.csv", index=False)

# Top 3 by composite score
if not df_scores.empty:
    top_3_save = df_scores.sort_values("Score", ascending=False).head(3).copy()
    top_3_save["params"] = top_3_save["params"].apply(str)
    top_3_save.to_csv("Data/top_3_v2.csv", index=False)
    print("Saved: top_3_v2.csv")

# BTC daily OHLC
from src.data_loader import load_btc_daily
df_btc_daily = load_btc_daily(config)
df_btc_daily.to_csv("Data/btc_daily_ohlc.csv", index=False)

print(f"Saved: results_v2_sweep.pkl ({len(results)} combos)")
print(f"Saved: sweep_IS_v2.csv ({len(df_summary)} rows)")
print(f"Saved: btc_daily_ohlc.csv ({len(df_btc_daily)} rows)")